<div style="display: flex; justify-content: space-between; align-items: center;">
    <div style="text-align: left; flex: 4">
        <strong>Author:</strong> Amirhossein Heydari — 
        📧 <a href="mailto:amirhosseinheydari78@gmail.com">amirhosseinheydari78@gmail.com</a> — 
        🐙 <a href="https://github.com/mr-pylin/pandas-workshop" target="_blank" rel="noopener">github.com/mr-pylin</a>
    </div>
    <div style="text-align: right; flex: 1;">
        <a href="https://pandas.pydata.org/" target="_blank" rel="noopener noreferrer">
            <img src="../assets/images/pandas/logo/pandas_white.svg" 
                 alt="Pandas Logo"
                 style="max-height: 48px; width: auto; background-color: #1f1f1f; border-radius: 8px;">
        </a>
    </div>
</div>
<hr>


**Table of contents**<a id='toc0_'></a>    
- [Dependencies](#toc1_)    
- [Load Sales Dataset](#toc2_)    
- [Aggregation and Grouping](#toc3_)    
  - [GroupBy Fundamentals](#toc3_1_)    
    - [The `groupby()` operation](#toc3_1_1_)    
    - [Grouping by columns, indexes, and custom keys](#toc3_1_2_)    
    - [Exploring group objects](#toc3_1_3_)    
  - [Aggregation](#toc3_2_)    
    - [Common aggregation functions (sum, mean, count, etc.)](#toc3_2_1_)    
    - [Single-function aggregations](#toc3_2_2_)    
    - [Applying multiple aggregations](#toc3_2_3_)    
    - [Custom aggregation functions](#toc3_2_4_)    
  - [Transformation and Filtering](#toc3_3_)    
    - [Using `.transform()` for aligned operations](#toc3_3_1_)    
    - [Conditional filtering with `.filter()`](#toc3_3_2_)    
    - [Combining aggregation, transformation, and filtering](#toc3_3_3_)    
  - [Multi-Level Grouping](#toc3_4_)    
    - [Grouping by multiple columns or index level](#toc3_4_1_)    
    - [Working with hierarchical indexes after grouping](#toc3_4_2_)    
  - [Window and Rolling Operations](#toc3_5_)    
    - [Rolling statistics (moving average, std, etc.)](#toc3_5_1_)    
    - [Expanding and exponentially weighted windows (`expanding`, `ewm`)](#toc3_5_2_)    
    - [Grouped rolling operations](#toc3_5_3_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# <a id='toc1_'></a>[Dependencies](#toc0_)


In [ ]:
import pandas as pd

In [ ]:
# disable wrapping entirely
pd.set_option("display.expand_frame_repr", False)

# <a id='toc2_'></a>[Load Sales Dataset](#toc0_)


In [ ]:
SALES_PATH = r"https://raw.githubusercontent.com/mr-pylin/datasets/refs/heads/main/data/tabular-data/sales/dataset.csv"
sales_df = pd.read_csv(SALES_PATH, encoding="latin1")

In [ ]:
# drop columns that are not needed for this topic
cols_to_drop = [
    "ORDERLINENUMBER",
    "CUSTOMERNAME",
    "PHONE",
    "ADDRESSLINE1",
    "ADDRESSLINE2",
    "CITY",
    "STATE",
    "POSTALCODE",
    "COUNTRY",
    "TERRITORY",
    "CONTACTLASTNAME",
    "CONTACTFIRSTNAME",
]

sales_df.drop(columns=cols_to_drop, inplace=True)

In [ ]:
# reorder columns for clarity
cols_order = [
    "ORDERNUMBER",
    "ORDERDATE",
    "YEAR_ID",
    "QTR_ID",
    "MONTH_ID",
    "PRODUCTLINE",
    "PRODUCTCODE",
    "DEALSIZE",
    "STATUS",
    "QUANTITYORDERED",
    "PRICEEACH",
    "MSRP",
    "SALES",
]

sales_df = sales_df[cols_order]

In [ ]:
sales_df.head()

In [ ]:
sales_df.info()

# <a id='toc3_'></a>[Aggregation and Grouping](#toc0_)

<div style="text-align: center; padding-top: 10px;">
    <img src="../assets/images/pandas/tutorials/06/groupby_select_detail.svg" alt="Aggregation Example" style="min-width: 256px; max-height: 40%; width: auto; background-color: #DBDBDB; border-radius: 16px;">
    <p><em>Figure 1: Aggregation example on titanic dataset</em> (<a href="https://pandas.pydata.org/docs/getting_started/intro_tutorials/" target="_blank">source</a>)</p>
</div>


## <a id='toc3_1_'></a>[GroupBy Fundamentals](#toc0_)


### <a id='toc3_1_1_'></a>[The `groupby()` operation](#toc0_)

- The `groupby` operation is at the heart of **grouped data analysis** in pandas.
- The workflow is often called **“split-apply-combine”**:
  1. **Split** the data into groups based on keys or columns.
  1. **Apply** a function (aggregation, transformation, or filtering) to each group.
  1. **Combine** the results into a new DataFrame or Series.

<div style="text-align: center; padding-top: 10px;">
    <img src="../assets/images/pandas/tutorials/06/groupby.svg" alt="grouped by" style="min-width: 256px; max-height: 40%; width: auto; background-color: #DBDBDB; border-radius: 16px;">
    <p><em>Figure 2: Aggregating statistics grouped by category</em> (<a href="https://pandas.pydata.org/docs/getting_started/intro_tutorials/" target="_blank">source</a>)</p>
</div>

💡 **Example Use Cases**
- Total sales per region
- Average score per class
- Number of orders per customer

💡 **Key Notes:**
- The resulting object is a `DataFrameGroupBy` or `SeriesGroupBy`.
- No computations are done until an **aggregation or transformation** is applied.

📝 **Docs**:
- `pandas.DataFrame.groupby`: [pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html)
- Group by: split-apply-combine: [pandas.pydata.org/docs/user_guide/groupby.html](https://pandas.pydata.org/docs/user_guide/groupby.html)


In [ ]:
# get unique values for a column
sales_df["PRODUCTLINE"].unique().tolist()

In [ ]:
# split: group by PRODUCTLINE
grouped = sales_df.groupby("PRODUCTLINE")

# log
print(f"type(grouped)  : {type(grouped)}")
print(f"grouped.groups : {grouped.groups}")
grouped.get_group("Classic Cars")

In [ ]:
# apply: perform aggregation on each group
total_sales_per_line = grouped["SALES"].sum()

# log
print(total_sales_per_line)

In [ ]:
# another example: number of orders per deal size
orders_per_dealsize = sales_df.groupby("DEALSIZE")["ORDERNUMBER"].nunique()
print(orders_per_dealsize)

### <a id='toc3_1_2_'></a>[Grouping by columns, indexes, and custom keys](#toc0_)

Pandas `groupby` allows flexible **grouping strategies** to organize data according to your analysis needs.

- **Grouping by columns:**
  - Most common approach.
  - Example: `df.groupby('Region')` groups rows by the `Region` column.

- **Grouping by index:**
  - Use the row index as the key.
  - Example: `df.groupby(df.index)`

- **Grouping by multiple columns:**
  - Groups based on **unique combinations** of multiple columns.
  - Example: `df.groupby(['Region', 'Product'])`

- **Custom grouping keys:**
  - Pass a **function**, **dict**, or **Series** to define group membership.
  - Example: group ages into categories: `df.groupby(lambda x: x // 10)`

💡 **Tip:**
> You can mix column names and custom keys to create complex groupings.

📝 **Docs**:
- `pandas.DataFrame.groupby`: [pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html)


In [ ]:
# grouping by a single column
grouped_productline = sales_df.groupby("PRODUCTLINE")["SALES"].sum()

# log
grouped_productline

In [ ]:
# grouping by index
# for demonstration, group by whether the index is even or odd
grouped_index = sales_df.groupby(lambda x: "even" if x % 2 == 0 else "odd")["SALES"].sum()

# log
grouped_index

In [ ]:
# grouping by multiple columns
grouped_multi = sales_df.groupby(["PRODUCTLINE", "DEALSIZE"])["SALES"].sum()

# log
grouped_multi

In [ ]:
# custom grouping keys
# categorize sales as 'Low', 'Medium', 'High'
def sales_category(sales):
    if sales < 1000:
        return "Low"
    elif sales < 5000:
        return "Medium"
    else:
        return "High"


grouped_custom = sales_df.groupby(sales_df["SALES"].apply(sales_category))["SALES"].sum()

# log
grouped_custom

### <a id='toc3_1_3_'></a>[Exploring group objects](#toc0_)

After creating a `GroupBy` object, you can **inspect and explore** its structure before applying computations.

- **Common attributes and methods:**
  - `.groups` – Dictionary mapping **group names** to **row labels**.
  - `.size()` – Returns the **number of rows** in each group.
  - `.get_group(name)` – Retrieves the **DataFrame or Series** for a specific group.

💡 **Tip:**
> Exploring groups helps ensure your data is correctly segmented before aggregation or transformation.

📝 **Docs**:
- `pandas.core.groupby.GroupBy`: [pandas.pydata.org/docs/reference/api/pandas.core.groupby.GroupBy.html](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.GroupBy.html)


In [ ]:
# create a GroupBy object by PRODUCTLINE
grouped = sales_df.groupby("PRODUCTLINE")

In [ ]:
# .groups: dictionary mapping group names to row indices
grouped.groups

In [ ]:
# .size(): number of rows in each group
group_sizes = grouped.size()
group_sizes

In [ ]:
# .get_group(name): retrieve the DataFrame for a specific group
motorcycles_group = grouped.get_group("Motorcycles")
motorcycles_group.head()

## <a id='toc3_2_'></a>[Aggregation](#toc0_)

- Aggregation is the process of **combining multiple values** into a **single summary value**.
- It is a core concept in data analysis and is used to extract **high-level insights** from raw data.
- Aggregation can be applied to a **Series**, **DataFrame**, or after a **grouping operation**.

<div style="text-align: center; padding-top: 10px;">
    <img src="../assets/images/pandas/tutorials/06/aggregate.svg" alt="Aggregation" style="min-width: 256px; max-height: 40%; width: auto; background-color: #DBDBDB; border-radius: 16px;">
    <p><em>Figure 3: Aggregate over a single column</em> (<a href="https://pandas.pydata.org/docs/getting_started/intro_tutorials/" target="_blank">source</a>)</p>
</div>
<div style="text-align: center; padding-top: 10px;">
    <img src="../assets/images/pandas/tutorials/06/reduction.svg" alt="Aggregation" style="min-width: 256px; max-height: 40%; width: auto; background-color: #DBDBDB; border-radius: 16px;">
    <p><em>Figure 4: Aggregate over multiple columns</em> (<a href="https://pandas.pydata.org/docs/getting_started/intro_tutorials/" target="_blank">source</a>)</p>
</div>

💡 **Example Scenarios**
- Calculating the **total sales** per store.
- Finding the **average test score** per class.
- Counting the **number of orders** per customer.

📝 **Docs**:
- `pandas.DataFrame.aggregate`: [pandas.pydata.org/docs/reference/api/pandas.DataFrame.aggregate.html](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.aggregate.html)
- `pandas.Series.aggregate`: [pandas.pydata.org/docs/reference/api/pandas.Series.aggregate.html](https://pandas.pydata.org/docs/reference/api/pandas.Series.aggregate.html)
- `pandas.DataFrame.agg`: [pandas.pydata.org/docs/reference/api/pandas.DataFrame.agg.html](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.agg.html)
- `pandas.Series.agg`: [pandas.pydata.org/docs/reference/api/pandas.Series.agg.html](https://pandas.pydata.org/docs/reference/api/pandas.Series.agg.html)
- Aggregation: [pandas.pydata.org/docs/user_guide/groupby.html#aggregation](https://pandas.pydata.org/docs/user_guide/groupby.html#aggregation)


### <a id='toc3_2_1_'></a>[Common aggregation functions (sum, mean, count, etc.)](#toc0_)

- **`sum()`** – Adds up all values.
- **`mean()`** – Computes the arithmetic average.
- **`count()`** – Counts the number of non-NA/null entries.
- **`min()` / `max()`** – Finds the smallest or largest value.
- **`median()`** – Returns the median value.
- **`std()` / `var()`** – Computes standard deviation or variance.

<div style="text-align: center; padding-top: 10px;">
    <img src="../assets/images/pandas/tutorials/06/valuecounts.svg" alt="value counts" style="min-width: 256px; max-height: 40%; width: auto; background-color: #DBDBDB; border-radius: 16px;">
    <p><em>Figure 5: Count number of records by category</em> (<a href="https://pandas.pydata.org/docs/getting_started/intro_tutorials/" target="_blank">source</a>)</p>
</div>

💡 **Usage Tip:**
> You can apply these functions individually or pass a **list/dict of functions** for multiple aggregations at once.

📝 **Docs**:
- `pandas.DataFrame.sum`: [pandas.pydata.org/docs/reference/api/pandas.DataFrame.sum.html](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sum.html)
- `pandas.DataFrame.mean`: [pandas.pydata.org/docs/reference/api/pandas.DataFrame.mean.html](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.mean.html)
- `pandas.DataFrame.count`: [pandas.pydata.org/docs/reference/api/pandas.DataFrame.count.html](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.count.html)
- `pandas.DataFrame.min`: [pandas.pydata.org/docs/reference/api/pandas.DataFrame.min.html](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.min.html)
- `pandas.DataFrame.max`: [pandas.pydata.org/docs/reference/api/pandas.DataFrame.max.html](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.max.html)
- `pandas.DataFrame.median`: [pandas.pydata.org/docs/reference/api/pandas.DataFrame.median.html](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.median.html)
- `pandas.DataFrame.std`: [pandas.pydata.org/docs/reference/api/pandas.DataFrame.std.html](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.std.html)
- `pandas.DataFrame.var`: [pandas.pydata.org/docs/reference/api/pandas.DataFrame.var.html](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.var.html)


### <a id='toc3_2_2_'></a>[Single-function aggregations](#toc0_)

- Single-function aggregation applies **one aggregation function** to a group or column.
- Use single-function aggregations when you need **one metric per group**.


In [ ]:
# group by PRODUCTLINE
grouped = sales_df.groupby("PRODUCTLINE")

In [ ]:
# total sales per product line
total_sales = grouped["SALES"].sum()

# log
total_sales

In [ ]:
# average quantity ordered per product line
avg_quantity = grouped["QUANTITYORDERED"].mean()

# log
avg_quantity

In [ ]:
# maximum quantity ordered per product line
max_price = grouped["QUANTITYORDERED"].max()

# log
max_price

### <a id='toc3_2_3_'></a>[Applying multiple aggregations](#toc0_)

- Pandas allows applying **multiple aggregation functions** to a group at once, providing a richer summary.
- Named aggregations (`agg(new_name=('column','func'))`) make output **cleaner and more readable**.


In [ ]:
# group by PRODUCTLINE
grouped = sales_df.groupby("PRODUCTLINE")

In [ ]:
# multiple aggregations using a list of functions
multi_agg = grouped["SALES"].aggregate(["sum", "mean", "min", "max", "std"])

# log
multi_agg

In [ ]:
# multiple aggregations using a dict of functions
multi_agg = grouped.aggregate({"SALES": ["sum", "mean", "min", "max", "std"]})

# log
multi_agg

In [ ]:
# named aggregations for cleaner output
named_agg = grouped.aggregate(
    total_sales=("SALES", "sum"),
    average_sales=("SALES", "mean"),
    max_price=("PRICEEACH", "max"),
    avg_quantity=("QUANTITYORDERED", "mean"),
)

# log
named_agg

### <a id='toc3_2_4_'></a>[Custom aggregation functions](#toc0_)

- Pandas allows the use of **custom functions** for aggregation, giving full flexibility to compute **specific metrics**.
- Custom functions can be **any Python function** that takes a Series and returns a scalar value.


In [ ]:
# group by PRODUCTLINE
grouped = sales_df.groupby("PRODUCTLINE")

In [ ]:
# define custom aggregation: range of sales (max - min)
def sales_range(x):
    return x.max() - x.min()


# apply custom aggregation to SALES
custom_agg = grouped["SALES"].aggregate(sales_range)

# log
custom_agg

In [ ]:
# define custom aggregation using multiple columns
# weighted average price = sum(PRICEEACH * QUANTITYORDERED) / sum(QUANTITYORDERED)
def weighted_avg_price(df):
    return (df["PRICEEACH"] * df["QUANTITYORDERED"]).sum() / df["QUANTITYORDERED"].sum()

# apply custom aggregation
weighted_avg = grouped.apply(weighted_avg_price, include_groups=False)

# log
weighted_avg

## <a id='toc3_3_'></a>[Transformation and Filtering](#toc0_)


### <a id='toc3_3_1_'></a>[Using `.transform()` for aligned operations](#toc0_)

- The `.transform()` method allows you to **perform operations on groups** while **maintaining the original DataFrame shape**.
- Use `.transform()` when you need **group-aware computations** but want the result **aligned with the original rows**.

✍️ **Key Features:**
  - Returns a Series or DataFrame **with the same index** as the original.
  - Often used for **standardization, normalization, or deviations from group statistics**.

📝 **Docs**:
- `pandas.DataFrame.transform`: [pandas.pydata.org/docs/reference/api/pandas.DataFrame.transform.html](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.transform.html)


In [ ]:
# group by PRODUCTLINE
grouped = sales_df.groupby("PRODUCTLINE")

In [ ]:
# normalize SALES within each group (aligned with original rows)
sales_df["SALES_normalized"] = grouped["SALES"].transform(lambda x: x / x.sum())

# log
sales_df[["PRODUCTLINE", "SALES", "SALES_normalized"]].head()

In [ ]:
# compute deviation from group mean
sales_df["SALES_dev_from_mean"] = grouped["SALES"].transform(lambda x: x - x.mean())

# log
sales_df[["PRODUCTLINE", "SALES", "SALES_dev_from_mean"]].head()

In [ ]:
# standardization (z-score) per group
sales_df["SALES_zscore"] = grouped["SALES"].transform(lambda x: (x - x.mean()) / x.std())

# log
sales_df[["PRODUCTLINE", "SALES", "SALES_zscore"]].head()

### <a id='toc3_3_2_'></a>[Conditional filtering with `.filter()`](#toc0_)

- The `.filter()` method allows you to **select groups based on a condition**, keeping or discarding entire groups.
- `.filter()` is useful for **removing small or irrelevant groups** before further analysis.

✍️ **Key Features:**
  - Returns a DataFrame containing **only groups that meet the condition**.
  - The function passed to `.filter()` receives a **group DataFrame** and should return **True** or **False**.

📝 **Docs**:
- `pandas.DataFrame.filter`: [pandas.pydata.org/docs/reference/api/pandas.DataFrame.filter.html](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.filter.html)


In [ ]:
# group by PRODUCTLINE
grouped = sales_df.groupby("PRODUCTLINE")

In [ ]:
# keep only product lines with total SALES > 1,000,000
high_sales_groups = grouped.filter(lambda x: x["SALES"].sum() > 1000000)

# log
high_sales_groups["PRODUCTLINE"].unique()

In [ ]:
# keep groups with at least 100 orders
large_groups = grouped.filter(lambda x: x["ORDERNUMBER"].nunique() >= 100)

# log
large_groups["PRODUCTLINE"].unique()

In [ ]:
# combine with a custom condition: SALES mean > 3500
custom_filtered = grouped.filter(lambda x: x["SALES"].mean() > 3500)

# log
custom_filtered["PRODUCTLINE"].unique()

### <a id='toc3_3_3_'></a>[Combining aggregation, transformation, and filtering](#toc0_)

- Pandas allows you to **combine aggregation, transformation, and filtering** to perform **complex group-based analyses**.
- This combination allows you to **clean, normalize, and summarize** grouped data efficiently.

✍️ **Workflow Example:**
  1. **Filter groups** based on a condition.
  1. **Transform values** within the remaining groups.
  1. **Aggregate** results to compute summary metrics.

📝 **Docs**:
- `pandas.DataFrame.groupby`: [pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html)


In [ ]:
# group by PRODUCTLINE
grouped = sales_df.groupby("PRODUCTLINE")

In [ ]:
# step 1: filter groups (keep product lines with total SALES > 200,000)
filtered = grouped.filter(lambda x: x["SALES"].sum() > 200000)

# step 2: transform values within remaining groups (normalize SALES per group)
filtered["SALES_normalized"] = filtered.groupby("PRODUCTLINE")["SALES"].transform(lambda x: x / x.sum())

# step 3: aggregate to compute summary metrics (mean and max of normalized SALES)
summary = filtered.groupby("PRODUCTLINE")["SALES_normalized"].aggregate(["mean", "max"])

# log
summary

## <a id='toc3_4_'></a>[Multi-Level Grouping](#toc0_)

- Multi-level grouping allows you to **analyze data across multiple dimensions**.
- Results often produce a **MultiIndex DataFrame**, which can be further manipulated using **named aggregations** or **index operations**.

💡 **Use Cases:**
- Total sales per **region and product category**
- Average scores per **class and gender**
- Revenue per **year and quarter**

📝 **Docs**:
- `pandas.DataFrame.groupby`: [pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html)
- MultiIndex / advanced indexing: [pandas.pydata.org/docs/user_guide/advanced.html#hierarchical-indexing](https://pandas.pydata.org/docs/user_guide/advanced.html#hierarchical-indexing)


### <a id='toc3_4_1_'></a>[Grouping by multiple columns or index level](#toc0_)


In [ ]:
# grouping by multiple columns
multi_grouped = sales_df.groupby(["PRODUCTLINE", "DEALSIZE"])

# compute total sales for each combination
total_sales_multi = multi_grouped["SALES"].sum()

# log
total_sales_multi

In [ ]:
# grouping by index level (example: row index parity)
index_grouped = sales_df.groupby(lambda x: "even" if x % 2 == 0 else "odd")

# compute total sales for each combination
total_sales_index = index_grouped["SALES"].sum()

# log
total_sales_index

### <a id='toc3_4_2_'></a>[Working with hierarchical indexes after grouping](#toc0_)

- After multi-level grouping, pandas produces a **MultiIndex DataFrame or Series**.
- Working with MultiIndex effectively allows **hierarchical slicing, selection, and aggregation**.
- You can manipulate this hierarchical index for **cleaner analysis**.


In [ ]:
# multi-level grouping by PRODUCTLINE and DEALSIZE
multi_grouped = sales_df.groupby(["PRODUCTLINE", "DEALSIZE"])["SALES"].sum()

# log
multi_grouped.head()

In [ ]:
# access data for a specific combination
motorcycles_large = multi_grouped.loc[("Motorcycles", "Large")]

# log
motorcycles_large

In [ ]:
# access all DEALSIZE for a specific PRODUCTLINE
motorcycles_all = multi_grouped.loc["Motorcycles"]

# log
motorcycles_all

In [ ]:
# reset index for a cleaner DataFrame
multi_grouped_df = multi_grouped.reset_index()

# log
multi_grouped_df.head()

## <a id='toc3_5_'></a>[Window and Rolling Operations](#toc0_)


In [ ]:
# object to datetime conversion
sales_df["ORDERDATE"] = pd.to_datetime(sales_df["ORDERDATE"])

# ensure data is sorted by ORDERDATE
sales_df_sorted = sales_df.sort_values("ORDERDATE")

### <a id='toc3_5_1_'></a>[Rolling statistics (moving average, std, etc.)](#toc0_)

- Rolling statistics compute **metrics over a sliding window**, allowing you to capture **local patterns** in the data.
- Rolling statistics are especially useful in **time series analysis** to smooth out short-term fluctuations.

✍️ **Key Concepts:**
  - A window can be **fixed-size** (rolling) or **expanding**.
  - Functions like `sum()`, `mean()`, `max()`, and `min()` can be applied within each window.
  - Useful for **time series, smoothing, and trend analysis**.

✍️ **Common rolling metrics:**
  - `mean()` – Moving average
  - `sum()` – Moving sum
  - `std()` – Moving standard deviation
  - `min()` / `max()` – Moving min/max

📝 **Docs**:
- `pandas.DataFrame.rolling`: [pandas.pydata.org/docs/reference/api/pandas.DataFrame.rolling.html](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rolling.html)


In [ ]:
# define a rolling window of 5 orders
window_size = 5
rolling_sales = sales_df_sorted["SALES"].rolling(window=window_size)

# rolling mean (moving average)
sales_df_sorted["SALES_roll_mean"] = rolling_sales.mean()

# rolling standard deviation
sales_df_sorted["SALES_roll_std"] = rolling_sales.std()

# log
sales_df_sorted[["ORDERDATE", "SALES", "SALES_roll_mean", "SALES_roll_std"]].head(10)

### <a id='toc3_5_2_'></a>[Expanding and exponentially weighted windows (`expanding`, `ewm`)](#toc0_)

- Pandas provides **expanding** and **exponentially weighted windows** for advanced rolling computations.
- Use expanding for **cumulative metrics** and ewm for **trend detection emphasizing recent data**.
- **Expanding windows:**
  - Include **all rows from the start** up to the current row.
- **Exponentially weighted windows (`ewm`):**
  - Apply **more weight to recent observations**.

📝 **Docs**:
- `pandas.DataFrame.expanding`: [pandas.pydata.org/docs/reference/api/pandas.DataFrame.expanding.html](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.expanding.html)
- `pandas.DataFrame.ewm`: [pandas.pydata.org/docs/reference/api/pandas.DataFrame.ewm.html](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.ewm.html)


In [ ]:
# expanding window: cumulative sum of SALES
# cumulative mean of SALES
sales_df_sorted["SALES_expanding_mean"] = sales_df_sorted["SALES"].expanding().mean()

# cumulative sum of SALES
sales_df_sorted["SALES_expanding_sum"] = sales_df_sorted["SALES"].expanding().sum()

# log
sales_df_sorted[["ORDERDATE", "SALES", "SALES_expanding_mean", "SALES_expanding_sum"]].head(10)

### <a id='toc3_5_3_'></a>[Grouped rolling operations](#toc0_)

- Pandas allows **rolling operations on grouped data**, enabling **windowed analysis within each group**.
- Grouped rolling operations are essential for **time series or sequential analysis per category**.

✍️ **Key Points:**
  - After `groupby`, `.rolling()` computes **windows within each group**.
  - Always **reset the index** if you want the result aligned with the original DataFrame.

📝 **Docs**:
- `pandas.DataFrame.rolling`: [pandas.pydata.org/docs/reference/api/pandas.DataFrame.rolling.html](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rolling.html)
- `pandas.DataFrame.groupby`: [pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html)


In [ ]:
# exponentially weighted mean (ewm) with span=3
sales_df_sorted["SALES_ewm"] = sales_df_sorted["SALES"].ewm(span=3, adjust=False).mean()

# exponentially weighted std (ewm) with span=3
sales_df_sorted["SALES_ewm_std"] = sales_df_sorted["SALES"].ewm(span=3, adjust=False).std()

# log
print(sales_df_sorted[["ORDERDATE", "SALES", "SALES_ewm", "SALES_ewm_std"]].head(10))